# 🧬 Universal Arise 4-Encoder Multi-Omics Runner (Google Colab)

This universal notebook provides a unified interface to run and compare both 4-Encoder model variants:

1. **Model Variant 1: `AriseSpatialGlue_4Encoder.py`**
   - **RNA Stream**: 3,000 Highly Variable Genes (HVG) input to **2-layer GCNs** (Spatial & Similarity).
   - **Aux Stream**: 1-layer GCNs (Spatial & Similarity).

2. **Model Variant 2: `AriseSpatialGlue_4Encoder_1Layer.py`**
   - **RNA Stream**: RNA expression reduced via **PCA (60 / 100 components)** input to **1-layer GCNs**.
   - **Aux Stream**: 1-layer GCNs.
   - **All 4 Encoders are 1-Layer GCNs**.

## 1. Environment & Dependencies Setup

In [ ]:
# Install required packages
!pip install -q scanpy anndata scikit-misc gdown torch_geometric
print("Dependencies successfully installed!")

## 2. 🎛️ Universal Experiment Configuration
Select your desired model architecture, datasets, seeds, epochs, and hyperparameters.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('models'))
sys.path.append(os.path.abspath('.'))

# ====================================================================
# ⚙️ EXPERIMENT SELECTION & HYPERPARAMETERS
# ====================================================================

# 1. Select Architecture Variant:
#    - 'standard' : RNA 3000 HVG with 2-Layer GCN (AriseSpatialGlue_4Encoder)
#    - '1layer_pca': RNA PCA (60/100 comps) with All 1-Layer GCNs (AriseSpatialGlue_4Encoder_1Layer)
MODEL_VARIANT = '1layer_pca'   # Options: 'standard' or '1layer_pca'

# 2. RNA PCA Components (Only used when MODEL_VARIANT == '1layer_pca'):
RNA_PCA_COMPS = 60            # e.g. 60 or 100

# 3. Select Datasets:
#    0: '10x_human_lymph_node_A1'
#    1: '10x_human_lymph_node_D1'
#    2: 'Mouse_Brain_E11_S1'
#    3: 'Mouse_Brain_E13_S1'
#    4: 'Mouse_Brain_E15_S1'
#    5: 'Mouse_Brain_E18_S1'
SELECTED_DATASETS = [0]       # e.g. [0] or [0, 1, 2] or 'all'

# 4. Seeds to Run:
SEEDS = [42, 2024]            # e.g. [42, 2024] or None for all 20 seeds

# 5. Training Settings:
EPOCHS = 350                  # e.g. 350
LEARNING_RATE = 1e-3          # e.g. 0.001
BETA = 25.0                   # Reconstruction loss weight
GAMMA = 10.0                  # Spatial regularization weight
DELTA = 1.0                   # L1/L2 regularization weight
HIDDEN_DIM = 512              # Hidden dimension in GCN
OUT_DIM = 64                  # Output embedding size
OUTPUT_DIR = "results"
# ====================================================================

print(f"Selected Architecture: {MODEL_VARIANT.upper()}")
print(f"Datasets: {SELECTED_DATASETS} | Seeds: {SEEDS} | Epochs: {EPOCHS}")

## 3. 🚀 Run Experiment

In [ ]:
if MODEL_VARIANT == '1layer_pca':
    from AriseSpatialGlue_4Encoder_1Layer import run_experiment
    df_results = run_experiment(
        datasets=SELECTED_DATASETS,
        seeds=SEEDS,
        rna_pca_comps=RNA_PCA_COMPS,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        beta=BETA,
        gamma=GAMMA,
        delta=DELTA,
        hidden_dim=HIDDEN_DIM,
        out_dim=OUT_DIM,
        output_dir=OUTPUT_DIR
    )
else:
    from AriseSpatialGlue_4Encoder import run_experiment
    df_results = run_experiment(
        datasets=SELECTED_DATASETS,
        seeds=SEEDS,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        beta=BETA,
        gamma=GAMMA,
        delta=DELTA,
        hidden_dim=HIDDEN_DIM,
        out_dim=OUT_DIM,
        output_dir=OUTPUT_DIR
    )

# Display results dataframe
import pandas as pd
print("\n--- SUMMARY OF RESULTS ---")
print(df_results.to_string(index=False))

## 4. Option B: Run via CLI Commands in Terminal

In [ ]:
# Run Model Variant 2 (All 1-Layer with RNA PCA = 60):
!python models/AriseSpatialGlue_4Encoder_1Layer.py --datasets 0 --seeds 42 2024 --rna_pca_comps 60 --epochs 350

# Run Model Variant 1 (Standard 2-Layer RNA HVG 3000):
# !python models/AriseSpatialGlue_4Encoder.py --datasets 0 --seeds 42 2024 --epochs 350

## 5. Visualizations: Spatial Domains & UMAP

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Load a processed dataset and plot
dataset_name = "10x_human_lymph_node_A1"
if os.path.exists(f"data/{dataset_name}/adata_RNA.h5ad"):
    adata = sc.read_h5ad(f"data/{dataset_name}/adata_RNA.h5ad")
    adata.var_names_make_unique()
    print(f"Loaded {adata.n_obs} spots for dataset {dataset_name}.")
else:
    print("Run training cell first to download and process data.")